English language preprocessing for xtts_v2 

In [1]:
import os
import re
import unicodedata
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# --- CONFIGURATION ---
# Kaggle Input Paths
metadata_file = "/kaggle/input/karen-us-female-tts-dataset/metadata.csv"
wav_dir = "/kaggle/input/karen-us-female-tts-dataset/wav"

# Kaggle Output Paths
output_folder = "/kaggle/working/XTTS_Dataset"
processed_wav_dir = os.path.join(output_folder, "wavs")
os.makedirs(processed_wav_dir, exist_ok=True)

# Audio config
sample_rate = 24000  # XTTS native rate
top_db_trim = 25     # Silence trimming threshold

# --- TEXT CLEANING ---

abbreviations = {
    "mr.": "mister",
    "mrs.": "misses",
    "ms.": "miss",
    "dr.": "doctor",
    "prof.": "professor",
    "e.g": "for example",
    "i.e": "that is",
    "etc.": "and so on",
    "vs.": "versus",
    "st.": "street",
    "ave.": "avenue",
    "rd.": "road",
    "mt.": "mount",
    "lt.": "lieutenant",
    "capt.": "captain",
    "col.": "colonel",
    "gen.": "general",
    "sgt.": "sergeant",
    "sr.": "senior",
    "jr.": "junior",
    "a.m.": "morning",
    "p.m.": "evening",
    "inc.": "incorporated",
    "ltd.": "limited",
    "co.": "company",
    "corp.": "corporation"
}

def expand_abbreviations(text):
    for abbr, full in abbreviations.items():
        text = text.replace(abbr, full)
    return text
    
def normalize_unicode(text):
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = normalize_unicode(text).strip().lower()
    # Keep essential punctuation for XTTS prosody
    text = re.sub(r"[^a-zA-Z0-9.,!?'\s-]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

# --- LOAD & PROCESS ---
df = pd.read_csv(metadata_file, sep="|", names=["audio_id", "text"], quoting=3)
df = df.dropna(subset=["audio_id", "text"])

processed_data = []

print(f" Processing {len(df)} files for XTTS...")

for idx, row in tqdm(df.iterrows(), total=len(df)):
    wav_id = str(row['audio_id']).strip()
    text = str(row['text']).strip()
    
    # Standardize filename
    wav_filename = wav_id if wav_id.endswith(".wav") else wav_id + ".wav"
    input_path = os.path.join(wav_dir, wav_filename)
    
    if not os.path.exists(input_path):
        continue

    try:
        # Load and Resample
        y, sr = librosa.load(input_path, sr=sample_rate)
        
        # Trim silence and Normalize
        y, _ = librosa.effects.trim(y, top_db=top_db_trim)
        if len(y) > 0:
            y = y / (np.max(np.abs(y)) + 1e-9)
        
        # Save standardized WAV to Kaggle Working
        output_wav_name = f"{wav_id}.wav"
        output_wav_path = os.path.join(processed_wav_dir, output_wav_name)
        sf.write(output_wav_path, y, sample_rate)

        # Prepare for CSV (ID | Cleaned Text | Cleaned Text)
        cleaned = clean_text(text)
        processed_data.append((wav_id, cleaned, cleaned))
        
    except Exception as e:
        pass # Skip corrupted files

# --- 80/20 SPLIT & SAVE ---
print("\n Splitting dataset...")
train_data, eval_data = train_test_split(processed_data, test_size=0.20, random_state=42)

# Convert to DataFrames
train_df = pd.DataFrame(train_data)
eval_df = pd.DataFrame(eval_data)

# Save to LJSpeech format
train_df.to_csv(os.path.join(output_folder, "metadata_train.csv"), sep="|", index=False, header=False)
eval_df.to_csv(os.path.join(output_folder, "metadata_eval.csv"), sep="|", index=False, header=False)

print(f" Preprocessing Complete!")
print(f"Train samples: {len(train_df)} | Eval samples: {len(eval_df)}")

 Processing 6847 files for XTTS...


100%|██████████| 6847/6847 [04:06<00:00, 27.79it/s]



 Splitting dataset...
 Preprocessing Complete!
Train samples: 5417 | Eval samples: 1355
